# Notebook 07: GraphSAGE Training

Training a GraphSAGE model on DRKG for drug-disease link prediction. GraphSAGE is the primary model from XAIPath (Perdomo-Quinteiro et al., 2026), and this notebook extends it to DRKG with a focus on acute brain injury.

GraphSAGE learns node embeddings by aggregating features from sampled neighborhoods.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import SAGEConv
import csv
import os
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

/projectnb/chenggrp/Maha/drkg_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Building the graph

Loading DRKG triplets and construct a DGL heterogeneous graph.

In [2]:

df = pd.read_csv('../data/drkg.tsv', sep='\t', header=None, 
                  names=['source', 'relation', 'target'])


all_entities = pd.concat([df['source'], df['target']]).unique()
entity2id = {e: i for i, e in enumerate(all_entities)}
id2entity = {i: e for e, i in entity2id.items()}
num_entities = len(entity2id)


all_relations = df['relation'].unique()
relation2id = {r: i for i, r in enumerate(all_relations)}
num_relations = len(relation2id)

print(f"Entities: {num_entities:,}")
print(f"Relations: {num_relations:,}")
print(f"Triplets: {len(df):,}")

Entities: 97,238
Relations: 107
Triplets: 5,874,261


In [3]:

src = torch.tensor([entity2id[h] for h in df['source']])
dst = torch.tensor([entity2id[t] for t in df['target']])
rel = torch.tensor([relation2id[r] for r in df['relation']])


g = dgl.graph((src, dst), num_nodes=num_entities)
g.edata['rel'] = rel
g = dgl.add_self_loop(g)

print(f"Graph nodes: {g.num_nodes():,}")
print(f"Graph edges: {g.num_edges():,}")

Graph nodes: 97,238
Graph edges: 5,971,499


## Train/test split

In [4]:
treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]


train_pos = []
with open('../train/drkg_train.tsv') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            h, r, t = parts
            if r in treatment_relations and h in entity2id and t in entity2id:
                train_pos.append((entity2id[h], entity2id[t]))


test_pos = []
with open('../train/drkg_test.tsv') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            h, r, t = parts
            if r in treatment_relations and h in entity2id and t in entity2id:
                test_pos.append((entity2id[h], entity2id[t]))

train_pos = torch.tensor(train_pos)
test_pos = torch.tensor(test_pos)

print(f"Train positive edges: {len(train_pos):,}")
print(f"Test positive edges: {len(test_pos):,}")

Train positive edges: 49,414
Test positive edges: 2,684


## GraphSAGE Model

Two-layer GraphSAGE with mean aggregation. Outputs node embeddings that are used to score drug-disease pairs via dot product.

In [5]:
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim, aggregator_type='mean')
        self.conv2 = SAGEConv(hidden_dim, out_dim, aggregator_type='mean')
    
    def forward(self, blocks, x):
        x = F.relu(self.conv1(blocks[0], x))
        x = self.conv2(blocks[1], x)
        return x
    
    def score(self, h_emb, t_emb):
        return (h_emb * t_emb).sum(dim=-1)

embedding_dim = 400
model = GraphSAGE(embedding_dim, 256, 128).to(device)
node_features = nn.Embedding(num_entities, embedding_dim).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Node feature parameters: {sum(p.numel() for p in node_features.parameters()):,}")

Model parameters: 270,720
Node feature parameters: 38,895,200


## Training

In [ ]:
from dgl.dataloading import DataLoader, NeighborSampler, as_edge_prediction_sampler

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(node_features.parameters()), 
    lr=0.001
)

# v1 used: negative_sampler=dgl.dataloading.negative_sampler.Uniform(5)
# v1 used: train_edge_ids = torch.arange(train_g.num_edges()).to(device)
# v1 used: dataloader = DataLoader(train_g, ...)
# v2 (current): full graph with 64 negative samples

g = g.to(device)

sampler = NeighborSampler([10, 10])
sampler = as_edge_prediction_sampler(
    sampler,
    negative_sampler=dgl.dataloading.negative_sampler.Uniform(64)
)

train_edge_ids = torch.arange(g.num_edges()).to(device)
dataloader = DataLoader(
    g, train_edge_ids, sampler,
    batch_size=1024, shuffle=True, drop_last=False,
    num_workers=0
)

print(f"Training batches: {len(dataloader)}")

Training batches: 5832


In [ ]:
epochs = 5
model.train()
node_features.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (input_nodes, pos_graph, neg_graph, blocks) in enumerate(dataloader):
        x = node_features(input_nodes)
        h = model(blocks, x)
        
        pos_src, pos_dst = pos_graph.edges()
        neg_src, neg_dst = neg_graph.edges()
        
        h_pos_src = h[pos_src]
        h_pos_dst = h[pos_dst]
        h_neg_src = h[neg_src]
        h_neg_dst = h[neg_dst]
        
        pos_score = (h_pos_src * h_pos_dst).sum(dim=-1)
        neg_score = (h_neg_src * h_neg_dst).sum(dim=-1)
        
        # v1 used: loss = F.softplus(-pos_score).mean() + F.softplus(neg_score).mean()
        # v2 (current): margin ranking loss
        loss = F.relu(1.0 - pos_score.unsqueeze(1) + neg_score).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    print(f"Epoch {epoch+1} complete. Avg loss: {total_loss/len(dataloader):.4f}")

Epoch 1, Batch 0/5832, Loss: 65.9378
Epoch 1, Batch 50/5832, Loss: 12.6069
Epoch 1, Batch 100/5832, Loss: 5.5606
Epoch 1, Batch 150/5832, Loss: 3.8358
Epoch 1, Batch 200/5832, Loss: 2.7366
Epoch 1, Batch 250/5832, Loss: 2.5983
Epoch 1, Batch 300/5832, Loss: 1.9503
Epoch 1, Batch 350/5832, Loss: 1.6814
Epoch 1, Batch 400/5832, Loss: 1.4583
Epoch 1, Batch 450/5832, Loss: 1.4570
Epoch 1, Batch 500/5832, Loss: 1.3346
Epoch 1, Batch 550/5832, Loss: 0.9721
Epoch 1, Batch 600/5832, Loss: 1.0507
Epoch 1, Batch 650/5832, Loss: 0.9360
Epoch 1, Batch 700/5832, Loss: 0.8608
Epoch 1, Batch 750/5832, Loss: 0.7155
Epoch 1, Batch 800/5832, Loss: 0.6786
Epoch 1, Batch 850/5832, Loss: 0.6683
Epoch 1, Batch 900/5832, Loss: 0.6232
Epoch 1, Batch 950/5832, Loss: 0.5768
Epoch 1, Batch 1000/5832, Loss: 0.5825
Epoch 1, Batch 1050/5832, Loss: 0.4645
Epoch 1, Batch 1100/5832, Loss: 0.5063
Epoch 1, Batch 1150/5832, Loss: 0.4852
Epoch 1, Batch 1200/5832, Loss: 0.4356
Epoch 1, Batch 1250/5832, Loss: 0.4220
Epoch 1

## Generate node embeddings

In [8]:
model.eval()
node_features.eval()

with torch.no_grad():
    all_node_ids = torch.arange(num_entities).to(device)
    x = node_features(all_node_ids)
    x = F.relu(model.conv1(g, x))
    node_emb = model.conv2(g, x)

print(f"Node embedding shape: {node_emb.shape}")

Node embedding shape: torch.Size([97238, 128])


## Score candidate drugs against brain injury disease nodes

In [9]:
brain_injury_disease_list = [
    'Disease::MESH:D020521',
    'Disease::MESH:D002544',
    'Disease::MESH:D020300',
    'Disease::MESH:D020520',
    'Disease::MESH:D002538',
    'Disease::MESH:D001930',
    'Disease::MESH:D006470',
]

drug_list = []
with open("../drug_repurpose/infer_drug.tsv", newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t', fieldnames=['drug', 'ids'])
    for row in reader:
        drug_list.append(row['drug'])

drug_ids = [entity2id[d] for d in drug_list if d in entity2id]
disease_ids = [entity2id[d] for d in brain_injury_disease_list if d in entity2id]

print(f"Drugs: {len(drug_ids)}, Diseases: {len(disease_ids)}")

Drugs: 8104, Diseases: 7


In [10]:
with torch.no_grad():
    drug_embs = node_emb[torch.tensor(drug_ids).to(device)]
    
    all_scores = []
    all_ids = []
    
    for did in disease_ids:
        disease_emb = node_emb[did]
        scores = (drug_embs * disease_emb).sum(dim=-1).cpu().numpy()
        all_scores.append(scores)
        all_ids.append(np.array(drug_ids))
    
    all_scores = np.concatenate(all_scores)
    all_ids = np.concatenate(all_ids)

idx = np.argsort(all_scores)[::-1]
all_scores = all_scores[idx]
all_ids = all_ids[idx]

_, unique_indices = np.unique(all_ids, return_index=True)
topk_indices = np.sort(unique_indices)[:100]

graphsage_results = pd.DataFrame({
    'rank': range(1, 101),
    'drug': [id2entity[int(d)] for d in all_ids[topk_indices]],
    'score': all_scores[topk_indices]
})

graphsage_results.head(20)

,rank,drug,score
0,1,Compound::DB03717,15.971498
1,2,Compound::DB05421,15.119494
2,3,Compound::DB01597,14.888819
3,4,Compound::DB07081,14.745038
4,5,Compound::DB13724,14.663816
5,6,Compound::DB06439,14.642166
6,7,Compound::DB04257,14.544352
7,8,Compound::DB04038,14.482705
8,9,Compound::DB07178,14.440119
9,10,Compound::DB03602,14.411073


In [14]:
os.makedirs('../results', exist_ok=True)
graphsage_results.to_csv('../results/graphsage_v2_top100.csv', index=False)
torch.save(node_emb.cpu(), '../results/graphsage_v2_embeddings.pt')


In [15]:
import pubchempy as pcp

def get_drug_name(drugbank_id):
    try:
        results = pcp.get_compounds(drugbank_id, 'name')
        if results:
            syns = results[0].synonyms
            return syns[0] if syns else 'unknown'
        return 'unknown'
    except:
        return 'unknown'

graphsage_results['drug_name'] = graphsage_results['drug'].apply(
    lambda x: get_drug_name(x.replace('Compound::', ''))
)

print("GraphSAGE top 20:")
print(graphsage_results[['rank', 'drug_name', 'drug', 'score']].head(20).to_string(index=False))

GraphSAGE top 20:
 rank                                                                                                                                 drug_name              drug     score
    1                                                                   3-HYDROXY-4-(3,4,5-TRIHYDROXY-TETRAHYDRO-PYRAN-2-YLOXY)-PIPERIDIN-2-ONE Compound::DB03717 15.971498
    2                                                                                                                                 CP-122721 Compound::DB05421 15.119494
    3                                                                                                                                Cilastatin Compound::DB01597 14.888819
    4 (2r)-4-[(8r)-8-Methyl-2-(Trifluoromethyl)-5,6-Dihydro[1,2,4]triazolo[1,5-A]pyrazin-7(8h)-Yl]-4-Oxo-1-(2,4,5-Trifluorophenyl)butan-2-Amine Compound::DB07081 14.745038
    5                                                                                                                     

In [16]:
graphsage_results['drug_name'] = graphsage_results['drug'].apply(
    lambda x: get_drug_name(x.replace('Compound::', ''))
)
graphsage_results.to_csv('../results/graphsage_v2_top20_named.csv', index=False)

### Training (Configuration Notes)

Two configurations were tested and compared:

**v1 (original configuration)**
- Training data: treatment edges only (49,414 edges)
- Loss: softplus binary cross-entropy
- Negative samples: 5 per positive
- Final avg loss: 0.626
- Results: graphsage_top100.csv, graphsage_top20_named.csv
- Surfaces clinically interpretable candidates (Dabigatran, Decoglurant)
- MRR: 0.012

**v2 (retrained configuration)**  
- Training data: full DRKG graph (5,971,499 edges)
- Loss: margin ranking (margin=1.0)
- Negative samples: 64 per positive
- Final avg loss: 0.052
- Results: graphsage_v2_top100.csv, graphsage_v2_top20_named.csv
- MRR: 0.012